In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("mypdf.pdf")
data = loader.load()

c:\Users\lenovo\anaconda3\envs\test\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\lenovo\anaconda3\envs\test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs= text_splitter.split_documents(data)

print("Total number of chunks: ", len(docs))

Total number of chunks:  43


In [4]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from dotenv import load_dotenv
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector= embeddings.embed_query("hello world")
vector[:5]

c:\Users\lenovo\anaconda3\envs\test\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\lenovo\anaconda3\envs\test\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.ai.generativelanguage_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.ai.generativelanguage_v1beta past that date.
  warnings.warn(message, FutureWarning)


[-0.030033651739358902,
 0.004021478816866875,
 0.01552528329193592,
 -0.08638038486242294,
 -0.0035104958806186914]

In [5]:
vectorstore= Chroma.from_documents(documents=docs, embedding= GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

In [6]:
retriever= vectorstore.as_retriever(search_kwargs={"k":5}, search_type="similarity")

retrieved_docs= retriever.invoke("What is new in Developement of Multiple Combined Regression Methods for Rainfall Measurement paper?")

In [7]:
len(retrieved_docs)

5

In [8]:
print(retrieved_docs[0].page_content)

Development of Multiple Combined Regression
Methods for Rainfall Measurement.
Nusrat Jahan Prottasha 1, Md. Jashim Uddin 2, Md. Kowsher 3, Rokeya Khatun
Shorna4, Niaz Al Murshed 5, and Boktiar Ahmed Bappy 6
1 Daﬀodil International University Dhaka 1207, Bangladesh,
jahannusratprotta@gmail.com
2 Noakhali Science and Technology University, 3814, Dhaka,
mdjaud12@gmail.com
3 Stevens Institute of Technology, Hoboken, NJ 07030 USA,
ga.kowsher@gmail.com
4 Daﬀodil International University, 1207, Dhaka,
rokeyashorna5@gmail.com
5 Jahangirnagar University, 1342, Dhaka,
niazalmurshed.ai@gmail.com
6 Jhenaidah polytechnic institute, 7300, Dhaka,
entbappy73@gmail.com
Abstract. Rainfall forecast is imperative as overwhelming precipitation
can lead to numerous catastrophes. The prediction makes a diﬀerence for
individuals to require preventive measures. In addition, the expectation
ought to be precise. Most of the nations in the world is an agricultural


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, max_tokens=500)

In [10]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [11]:
system_prompt= (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [12]:
question_answer_chain= create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain= create_retrieval_chain(retriever,question_answer_chain)

In [13]:
response = rag_chain.invoke({"input": "What is new in Developement of Multiple Combined Regression Methods for Rainfall Measurement paper?"})
print(response["answer"])

The paper "Development of Multiple Combined Regression Methods for Rainfall Measurement" proposes using predictive regression analysis techniques to quantify the amount of rainfall in a specific location. Unlike many existing methods that focus on classification (whether it will rain or not), this work aims to predict the exact quantity of precipitation. It utilizes over 10 years of historical weather data from various places to train its models.
